# 04 - Fakturoid Integration

This notebook tests integration with Fakturoid API for submitting invoices.


In [1]:
from src.fakturoid_client import FakturoidClient
from src.config import config
import json


In [2]:
# Initialize Fakturoid client
fakturoid = FakturoidClient(config)

print("Fakturoid Client initialized")
print(f"Account: {config.fakturoid.account_slug}")
print(f"Base URL: {config.fakturoid.base_url}")


Fakturoid Client initialized
Account: bohemiafalconstudio
Base URL: https://app.fakturoid.cz/api/v3


In [3]:
# Test connection - get account info
try:
    account_info = fakturoid.get_account_info()
    print("✓ Successfully connected to Fakturoid")
    print(f"\nAccount Info:")
    print(json.dumps(account_info, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"✗ Failed to connect to Fakturoid: {e}")


✓ Successfully connected to Fakturoid

Account Info:
{
  "subdomain": "bohemiafalconstudio",
  "plan": "Na maximum",
  "plan_price": 519,
  "plan_paid_users": 0,
  "invoice_email": "zverina.pavel@seznam.cz",
  "phone": "",
  "web": "",
  "name": "Bohemia Falcon Studio, s.r.o.",
  "full_name": null,
  "registration_no": "24167185",
  "vat_no": "CZ24167185",
  "local_vat_no": null,
  "vat_mode": "vat_payer",
  "vat_price_mode": "without_vat",
  "street": "Plynární 1032/29",
  "city": "Praha 7",
  "zip": "17000",
  "country": "CZ",
  "currency": "CZK",
  "unit_name": "",
  "vat_rate": 21,
  "displayed_note": "Společnost je zapsána v obchodním rejstříku vedeném Městským soudem v Praze oddíl C, vložka 184888.",
  "invoice_note": null,
  "due": 14,
  "invoice_language": "cz",
  "invoice_payment_method": null,
  "invoice_proforma": false,
  "invoice_hide_bank_account_for_payments": null,
  "fixed_exchange_rate": false,
  "invoice_selfbilling": false,
  "default_estimate_type": null,
  "send_o

In [4]:
# List existing subjects (suppliers)
try:
    subjects = fakturoid.list_subjects()
    print(f"Found {len(subjects)} subjects in Fakturoid:")
    for subject in subjects[:5]:  # Show first 5
        print(f"  - {subject.get('name')} (ID: {subject.get('id')})")
    if len(subjects) > 5:
        print(f"  ... and {len(subjects) - 5} more")
except Exception as e:
    print(f"✗ Failed to list subjects: {e}")


Found 40 subjects in Fakturoid:
  - 123RF Limited (ID: 14468485)
  - 1. e-shop s.r.o. (ID: 14468465)
  - 1. holešovická restaurační s.r.o. (ID: 14468464)
  - 3D Magic LLC (ID: 14468551)
  - AB papír s.r.o. (ID: 14458962)
  ... and 35 more


In [5]:
# Test ARES lookup for Czech companies
print("Testing ARES (Czech Business Register) lookup")
print("="*60)

# Test with a real Czech company IČO
test_ico = "24167185"  # Bohemia Falcon Studio
print(f"\nLooking up IČO: {test_ico}")

ares_data = fakturoid.get_company_from_ares(test_ico)

if ares_data:
    print(f"\n✓ Company found in ARES:")
    print(f"  Name: {ares_data.get('name')}")
    print(f"  Street: {ares_data.get('street')}")
    print(f"  City: {ares_data.get('city')}")
    print(f"  ZIP: {ares_data.get('zip')}")
    print(f"  IČO: {ares_data.get('registration_no')}")
    print(f"  DIČ: {ares_data.get('vat_no')}")
else:
    print("\n✗ Company not found in ARES")

# Try another example
print(f"\n{'-'*60}")
test_ico2 = "27082440"  # Fakturoid s.r.o.
print(f"Looking up IČO: {test_ico2}")

ares_data2 = fakturoid.get_company_from_ares(test_ico2)
if ares_data2:
    print(f"\n✓ Company found in ARES:")
    print(f"  Name: {ares_data2.get('name')}")
    print(f"  Street: {ares_data2.get('street')}")
    print(f"  City: {ares_data2.get('city')}")


Testing ARES (Czech Business Register) lookup

Looking up IČO: 24167185

✓ Company found in ARES:
  Name: Bohemia Falcon Studio, s.r.o.
  Street: Plynární 1032/29
  City: Praha
  ZIP: 17000
  IČO: 24167185
  DIČ: CZ24167185

------------------------------------------------------------
Looking up IČO: 27082440

✓ Company found in ARES:
  Name: Alza.cz a.s.
  Street: Jankovcova 1522/53
  City: Praha


In [6]:
# Test invoice submission using submit_expense method
from src.ai_extractor import InvoiceData

# Create sample invoice data (using real Czech company for ARES test)
sample_invoice = InvoiceData(
    invoice_number="TEST-001",
    issue_date="2024-10-08",
    supplier_name="Fakturoid s.r.o.",  # Real company - ARES will update details
    total_amount=1210.0,  # Including 21% VAT
    due_date="2024-10-22",
    supplier_address="Testovací 123, Praha",  # Will be replaced by ARES data
    supplier_ico="27082440",  # Real IČO - Fakturoid s.r.o.
    supplier_dic="CZ27082440",
    currency="CZK",
    variable_symbol="001",
    notes="Test invoice for API integration"
)

print("Sample invoice data:")
print(json.dumps(sample_invoice.model_dump(), indent=2, ensure_ascii=False))

# Submit to Fakturoid (automatically creates supplier if needed)
print("\n" + "="*60)
print("SUBMITTING TO FAKTUROID...")
print("="*60)

try:
    result = fakturoid.submit_expense(sample_invoice, auto_create_subject=True)
    print(f"\n✓ Expense created successfully!")
    print(f"Expense ID: {result.get('id')}")
    print(f"Number: {result.get('number')}")
    print(f"Supplier: {result.get('supplier_name')}")
    print(f"Total: {result.get('total')} {result.get('currency')}")
    print(f"\nView in Fakturoid: {result.get('html_url')}")
except Exception as e:
    print(f"\n✗ Failed to submit: {e}")
    import traceback
    traceback.print_exc()


Sample invoice data:
{
  "invoice_number": "TEST-001",
  "issue_date": "2024-10-08",
  "supplier_name": "Fakturoid s.r.o.",
  "total_amount": 1210.0,
  "due_date": "2024-10-22",
  "variable_symbol": "001",
  "supplier_address": "Testovací 123, Praha",
  "supplier_street": null,
  "supplier_city": null,
  "supplier_zip": null,
  "supplier_country": null,
  "supplier_ico": "27082440",
  "supplier_dic": "CZ27082440",
  "supplier_vat_number": null,
  "currency": "CZK",
  "tax_amount": null,
  "line_items": null,
  "notes": "Test invoice for API integration",
  "confidence": null,
  "source_file": null
}

SUBMITTING TO FAKTUROID...
✓ Found existing subject by IČO: Alza.cz a.s.

🔍 DEBUG - Sending to Fakturoid API:
{
  "subject_id": 14468449,
  "lines": [
    {
      "name": "Test invoice for API integration",
      "quantity": "1.0",
      "unit_price": "1000.0",
      "vat_rate": 21
    }
  ],
  "original_number": "TEST-001",
  "variable_symbol": "001",
  "document_type": "invoice",
  "issu

In [7]:
# Test with FOREIGN supplier (detailed address extraction)
from src.ai_extractor import InvoiceData

print("Testing foreign supplier with detailed address")
print("="*60)

# Example: German company
foreign_invoice = InvoiceData(
    invoice_number="DE-2025-001",
    issue_date="2025-01-15",
    supplier_name="Example GmbH",
    total_amount=500.00,  # EUR
    due_date="2025-02-15",
    # Detailed address fields
    supplier_street="Hauptstraße 123",
    supplier_city="Berlin",
    supplier_zip="10115",
    supplier_country="DE",
    # VAT number for foreign company
    supplier_vat_number="DE123456789",
    currency="EUR",
    variable_symbol="2025001",
    notes="Test invoice from German supplier"
)

print("\nForeign invoice data:")
print(json.dumps(foreign_invoice.model_dump(), indent=2, ensure_ascii=False))

print(f"\n{'='*60}")
print("Ready to submit - UNCOMMENT to create expense")
print(f"{'='*60}")

# UNCOMMENT to actually submit:
try:
    result = fakturoid.submit_expense(foreign_invoice, auto_create_subject=True)
    print(f"\n✓ Expense created successfully!")
    print(f"Expense ID: {result.get('id')}")
    print(f"Number: {result.get('number')}")
    print(f"Supplier: {result.get('supplier_name')}")
    print(f"Total: {result.get('total')} {result.get('currency')}")
    print(f"\nView in Fakturoid: {result.get('html_url')}")
except Exception as e:
    print(f"\n✗ Failed to submit: {e}")
    import traceback
    traceback.print_exc()


Testing foreign supplier with detailed address

Foreign invoice data:
{
  "invoice_number": "DE-2025-001",
  "issue_date": "2025-01-15",
  "supplier_name": "Example GmbH",
  "total_amount": 500.0,
  "due_date": "2025-02-15",
  "variable_symbol": "2025001",
  "supplier_address": null,
  "supplier_street": "Hauptstraße 123",
  "supplier_city": "Berlin",
  "supplier_zip": "10115",
  "supplier_country": "DE",
  "supplier_ico": null,
  "supplier_dic": null,
  "supplier_vat_number": "DE123456789",
  "currency": "EUR",
  "tax_amount": null,
  "line_items": null,
  "notes": "Test invoice from German supplier",
  "confidence": null,
  "source_file": null
}

Ready to submit - UNCOMMENT to create expense
⚙ Creating new subject: Example GmbH
  ✓ Subject created with ID: 28429331

🔍 DEBUG - Sending to Fakturoid API:
{
  "subject_id": 28429331,
  "lines": [
    {
      "name": "Test invoice from German supplier",
      "quantity": "1.0",
      "unit_price": "413.22",
      "vat_rate": 21
    }
  ],


In [8]:
# List recent expense invoices
try:
    invoices = fakturoid.list_expense_invoices(limit=10)
    print(f"Recent expense invoices ({len(invoices)}):")
    for inv in invoices:
        print(f"  - {inv.get('number')} | {inv.get('supplier_name')} | {inv.get('total')} {inv.get('currency')}")
except Exception as e:
    print(f"✗ Failed to list invoices: {e}")


Recent expense invoices (40):
  - FP20250179 | OpenAI, LLC | 20.0 USD
  - FP20250178 | Google Cloud EMEA Limited | 7.78 USD
  - FP20250177 | Air Bank a.s. | 10.0 CZK
  - FP20250173 | Alza.cz a.s. | 2981.0 CZK
  - FP20250171 | Alza.cz a.s. | 159.0 CZK
  - FP20250170 | Apple Distribution International Ltd. | 129.0 CZK
  - FP20250176 | Facebook | 591.07 CZK
  - FP20250167 | Vodafone Czech Republic a.s. | 483.0 CZK
  - FP20250175 | Alza.cz a.s. | 165.0 CZK
  - FP20250174 | Alza.cz a.s. | 489.0 CZK
  - FP20250166 | Netflix International B.V. | 379.0 CZK
  - FP20250172 | Cursor | 24.2 USD
  - FP20250169 | Alza.cz a.s. | 250.0 CZK
  - FP20250165 | T-Mobile Czech Republic a.s. | 1836.5 CZK
  - FP20250164 | Apple Distribution International Ltd. | 249.0 CZK
  - FP20250163 | Fakturoid s.r.o. | 519.0 CZK
  - FP20250168 | MacParts.cz s. r. o. | 1688.0 CZK
  - FP20250162 | OpenAI, LLC | 20.0 USD
  - FP20250161 | Air Bank a.s. | 444.97 CZK
  - FP20250158 | Alibaba (China) Co., Ltd | 235.81 CZK
  - FP

In [9]:
# Test with REAL invoice from PDF
from src.ai_extractor import AIExtractor
from src.document_processor import DocumentProcessor

print("Testing with real invoice from PDF...")
print("="*60)

# Get first invoice file
doc_processor = DocumentProcessor(config.directories.invoices)
files = doc_processor.list_invoice_files()

if files:
    test_file = files[0]
    print(f"\n📄 Processing: {test_file.name}")
    
    # Extract data using AI
    ai_extractor = AIExtractor(config)
    invoice_data_dict = ai_extractor.extract_invoice_data(test_file)
    
    # Convert to InvoiceData model
    invoice_data = InvoiceData(**invoice_data_dict)
    
    print(f"\n✓ Extracted data:")
    print(f"  Supplier: {invoice_data.supplier_name}")
    print(f"  Invoice #: {invoice_data.invoice_number}")
    print(f"  Date: {invoice_data.issue_date}")
    print(f"  Amount: {invoice_data.total_amount} {invoice_data.currency}")
    
    # Ask before submitting
    print(f"\n{'='*60}")
    print("Ready to submit to Fakturoid")
    print(f"{'='*60}")
    
    # UNCOMMENT to actually submit:
    try:
        result = fakturoid.submit_expense(invoice_data, auto_create_subject=True)
        print(f"\n✓ Expense created successfully!")
        print(f"Expense ID: {result.get('id')}")
        print(f"Number: {result.get('number')}")
        print(f"Supplier: {result.get('supplier_name')}")
        print(f"Total: {result.get('total')} {result.get('currency')}")
        print(f"\nView in Fakturoid: {result.get('html_url')}")
    except Exception as e:
        print(f"\n✗ Failed to submit: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No invoice files found in data/invoices/")


Testing with real invoice from PDF...

📄 Processing: Alien Isolation.pdf

✓ Extracted data:
  Supplier: Sony Interactive Entertainment Network Europe Limited
  Invoice #: 786940972572357
  Date: 2025-10-02
  Amount: 207.25 CZK

Ready to submit to Fakturoid
⚙ Creating new subject: Sony Interactive Entertainment Network Europe Limited
  ✓ Subject created with ID: 28429344

🔍 DEBUG - Sending to Fakturoid API:
{
  "subject_id": 28429344,
  "lines": [
    {
      "name": "This is not a VAT/GST invoice. Email sent from a send-only address. Do not reply.",
      "quantity": "1.0",
      "unit_price": "171.28",
      "vat_rate": 21
    }
  ],
  "original_number": "786940972572357",
  "document_type": "invoice",
  "issued_on": "2025-10-02",
  "received_on": "2025-10-02",
  "description": "This is not a VAT/GST invoice. Email sent from a send-only address. Do not reply.",
  "currency": "CZK"
}

✓ Expense created successfully!
Expense ID: 3468032
Number: FP20250181
Supplier: Sony Interactive Ente